# Feature Engineering Pipeline — Daily Alarm Recurrences

This notebook builds two leakage-safe daily datasets for one-day-ahead forecasting. The target is the number of same-machine alarm recurrences with positive IAT below 30 minutes on the target day.

- **Simple model:** calendar features, causal gap indicators, and lagged total alarm volumes.
- **Complete model:** all simple features, the latest completed-month SELIC and durable-goods IPCA values, and lagged alarm volumes for TR10, TR17, and TR19.

In [13]:
import pandas as pd

from src.data_prep.db_preparation import load_db
from src.feat_eng.feature_eng import (
    TARGET_COL,
    build_feature_datasets,
    get_feature_columns,
    prepare_model_dataset,
    make_expanding_window_splits,
)

DATA_PATH = 'src/data/'
ALARMS_FILE = DATA_PATH + 'alarmescompletos.csv'
SELIC_FILE = DATA_PATH + 'selic2019a2025.csv'
IPCA_DURAVEIS_FILE = DATA_PATH + 'ipca_duraveis_serie.csv'

SIMPLE_FEATURES_OUTPUT = 'src/feat_eng/features_simple.csv'
FINAL_FEATURES_OUTPUT = 'src/feat_eng/features_final.csv'

## 1. Load the Alarm Events

In [14]:
df_alarms_raw = load_db(ALARMS_FILE)
print(f'Loaded {len(df_alarms_raw):,} alarm records from {ALARMS_FILE}')

Loaded 88,414 alarm records from src/data/alarmescompletos.csv


## 2. Build the Daily Target and Both Feature Tables

The row date is the day being predicted. Operational predictors are shifted by at least one day. Days with zero alarm records are retained in the intermediate daily base and marked as target-unavailable instead of being assumed to represent normal zero-recurrence operation.

In [15]:
feature_bundle = build_feature_datasets(
    df_alarms_raw,
    selic_path=SELIC_FILE,
    ipca_duraveis_path=IPCA_DURAVEIS_FILE,
    timestamp_col='TIMESTAMP',
    machine_col='Maq',
    iat_threshold_min=30,
    timezone='America/Recife',
)

daily_recurrence_base = feature_bundle['daily_base']
features_simple_all_days = feature_bundle['simple']
features_final_all_days = feature_bundle['complete']

display(daily_recurrence_base[[
    'Date', 'AlarmVolume', TARGET_COL, 'target_available'
]].head())
print(f"Total target recurrences: {daily_recurrence_base[TARGET_COL].sum():,}")
print(f"Unavailable/zero-record days: {(daily_recurrence_base['target_available'] == 0).sum():,}")

[LOG] Columns lowered:  ['shop', 'line', 'st', 'maq', 'bilancella', 'trato', 'fault_code1', 'fault_code2', 'fault_code3', 'posicao', 'timestamp']
[LOG] Duplicates removed: 3
[LOG] Columns stripped: ['shop', 'line', 'st', 'maq', 'bilancella', 'trato', 'fault_code1', 'fault_code2', 'fault_code3', 'posicao', 'timestamp']
[LOG] NaNs removed: 0
[LOG] Total lines: 88414 -> 88411 (3 lines removed)
[LOG] Inter-alarm time 'IAT_min' computed for 20 group(s): 88,391 valid interval(s), 0 non-positive interval(s) ignored
[LOG] Daily recurrence base built: 1,953 days, 1,527 target-available days, 31,712 recurrences below 30 minutes
[LOG] Temporal lags added for ['AlarmVolume']: ['1d', '3d', '7d', '14d', '21d', '28d']
[LOG] Simple feature table built: 1,953 rows x 19 columns
[LOG] SELIC loaded: 1400 daily records -> 67 months (0 unparsable values)
[LOG] IPCA de Duráveis loaded: 421 months, 0 missing values
[LOG] Available exogenous features added: SELIC and IPCA_Duraveis (lag=1 completed month(s))
[L

,Date,AlarmVolume,recurrences_lt_30m,target_available
0,2020-01-11 00:00:00-03:00,185,117,1
1,2020-01-12 00:00:00-03:00,69,20,1
2,2020-01-13 00:00:00-03:00,63,29,1
3,2020-01-14 00:00:00-03:00,102,42,1
4,2020-01-15 00:00:00-03:00,65,36,1


Total target recurrences: 31,712
Unavailable/zero-record days: 426


## 3. Prepare Model-Ready Datasets

Target-unavailable days and rows with incomplete lag history are excluded. Gap information remains available through strictly backward indicators such as `GapFlag_lag1d`, `GapRunLength_lag1d`, `GapDays_last7d`, and `GapDays_last30d`.

In [16]:
features_simple = prepare_model_dataset(features_simple_all_days)
features_final = prepare_model_dataset(features_final_all_days)

simple_feature_columns = get_feature_columns(features_simple)
final_feature_columns = get_feature_columns(features_final)

assert features_simple['Date'].is_monotonic_increasing
assert features_final['Date'].is_monotonic_increasing
assert features_simple[TARGET_COL].ge(0).all()
assert features_final[TARGET_COL].ge(0).all()
assert not any(column.startswith(('TR10_', 'TR17_', 'TR19_')) for column in simple_feature_columns)
assert all(
    any(column.startswith(f'{machine}_') for column in final_feature_columns)
    for machine in ('TR10', 'TR17', 'TR19')
)

print(f'Simple dataset: {len(features_simple):,} rows x {len(simple_feature_columns)} features')
print(f'Complete dataset: {len(features_final):,} rows x {len(final_feature_columns)} features')
display(features_simple.head())
display(features_final.head())

[LOG] Model dataset prepared: 1,499 rows x 16 features; 426 unavailable target day(s) excluded
[LOG] Model dataset prepared: 1,499 rows x 36 features; 426 unavailable target day(s) excluded
Simple dataset: 1,499 rows x 16 features
Complete dataset: 1,499 rows x 36 features


,Date,recurrences_lt_30m,target_available,Year,Month,Day,DayOfWeek,WeekOfYear,IsWeekend,GapFlag_lag1d,GapRunLength_lag1d,GapDays_last7d,GapDays_last30d,AlarmVolume_lag1d,AlarmVolume_lag3d,AlarmVolume_lag7d,AlarmVolume_lag14d,AlarmVolume_lag21d,AlarmVolume_lag28d
0,2020-02-08 00:00:00-03:00,46,1,2020,2,8,5,6,1,0.00,0.00,0.00,0.00,87.00,97.00,89.00,39.00,96.00,185.00
1,2020-02-09 00:00:00-03:00,1,1,2020,2,9,6,6,1,0.00,0.00,0.00,0.00,136.00,51.00,20.00,15.00,14.00,69.00
2,2020-02-10 00:00:00-03:00,5,1,2020,2,10,0,7,0,0.00,0.00,0.00,0.00,22.00,87.00,105.00,15.00,95.00,63.00
3,2020-02-11 00:00:00-03:00,70,1,2020,2,11,1,7,0,0.00,0.00,0.00,0.00,58.00,136.00,51.00,4.00,116.00,102.00
4,2020-02-12 00:00:00-03:00,13,1,2020,2,12,2,7,0,0.00,0.00,0.00,0.00,119.00,22.00,97.00,78.00,87.00,65.00


,Date,recurrences_lt_30m,target_available,Year,Month,Day,DayOfWeek,WeekOfYear,IsWeekend,GapFlag_lag1d,GapRunLength_lag1d,GapDays_last7d,GapDays_last30d,AlarmVolume_lag1d,AlarmVolume_lag3d,AlarmVolume_lag7d,AlarmVolume_lag14d,AlarmVolume_lag21d,AlarmVolume_lag28d,SELIC_available,IPCA_Duraveis_available,TR10_AlarmVolume_lag1d,TR10_AlarmVolume_lag3d,TR10_AlarmVolume_lag7d,TR10_AlarmVolume_lag14d,TR10_AlarmVolume_lag21d,TR10_AlarmVolume_lag28d,TR17_AlarmVolume_lag1d,TR17_AlarmVolume_lag3d,TR17_AlarmVolume_lag7d,TR17_AlarmVolume_lag14d,TR17_AlarmVolume_lag21d,TR17_AlarmVolume_lag28d,TR19_AlarmVolume_lag1d,TR19_AlarmVolume_lag3d,TR19_AlarmVolume_lag7d,TR19_AlarmVolume_lag14d,TR19_AlarmVolume_lag21d,TR19_AlarmVolume_lag28d
0,2020-02-08 00:00:00-03:00,46,1,2020,2,8,5,6,1,0.00,0.00,0.00,0.00,87.00,97.00,89.00,39.00,96.00,185.00,0.02,0.15,5.00,5.00,8.00,1.00,3.00,15.00,0.00,2.00,1.00,0.00,5.00,0.00,5.00,3.00,5.00,1.00,3.00,18.00
1,2020-02-09 00:00:00-03:00,1,1,2020,2,9,6,6,1,0.00,0.00,0.00,0.00,136.00,51.00,20.00,15.00,14.00,69.00,0.02,0.15,8.00,0.00,3.00,6.00,1.00,2.00,7.00,0.00,0.00,0.00,0.00,3.00,5.00,3.00,4.00,2.00,1.00,2.00
2,2020-02-10 00:00:00-03:00,5,1,2020,2,10,0,7,0,0.00,0.00,0.00,0.00,22.00,87.00,105.00,15.00,95.00,63.00,0.02,0.15,1.00,5.00,7.00,0.00,8.00,4.00,0.00,0.00,4.00,0.00,0.00,1.00,2.00,5.00,7.00,1.00,9.00,4.00
3,2020-02-11 00:00:00-03:00,70,1,2020,2,11,1,7,0,0.00,0.00,0.00,0.00,58.00,136.00,51.00,4.00,116.00,102.00,0.02,0.15,3.00,8.00,2.00,0.00,6.00,4.00,1.00,7.00,2.00,0.00,1.00,3.00,1.00,5.00,2.00,0.00,7.00,9.00
4,2020-02-12 00:00:00-03:00,13,1,2020,2,12,2,7,0,0.00,0.00,0.00,0.00,119.00,22.00,97.00,78.00,87.00,65.00,0.02,0.15,5.00,1.00,5.00,5.00,4.00,2.00,6.00,0.00,2.00,1.00,14.00,2.00,6.00,2.00,3.00,3.00,3.00,2.00


## 4. Expanding-Window Temporal Validation

Training always precedes validation. Each fold expands the historical training window and evaluates the following calendar period, preserving the non-stationary temporal structure identified during EDA.

In [17]:
simple_splits, simple_validation_summary = make_expanding_window_splits(
    features_simple,
    initial_train_days=730,
    validation_days=365,
    step_days=365,
)
final_splits, final_validation_summary = make_expanding_window_splits(
    features_final,
    initial_train_days=730,
    validation_days=365,
    step_days=365,
)

display(simple_validation_summary)
display(final_validation_summary)

[LOG] Expanding-window validation created: 4 fold(s)
[LOG] Expanding-window validation created: 4 fold(s)


,Fold,Train_start,Train_end,Validation_start,Validation_end,Train_rows,Validation_rows
0,1,2020-02-08 00:00:00-03:00,2022-02-06 00:00:00-03:00,2022-02-07 00:00:00-03:00,2023-02-06 00:00:00-03:00,585,250
1,2,2020-02-08 00:00:00-03:00,2023-02-06 00:00:00-03:00,2023-02-07 00:00:00-03:00,2024-02-06 00:00:00-03:00,835,214
2,3,2020-02-08 00:00:00-03:00,2024-02-06 00:00:00-03:00,2024-02-07 00:00:00-03:00,2025-02-05 00:00:00-03:00,1049,355
3,4,2020-02-08 00:00:00-03:00,2025-02-05 00:00:00-03:00,2025-02-06 00:00:00-03:00,2025-05-16 00:00:00-03:00,1404,95


,Fold,Train_start,Train_end,Validation_start,Validation_end,Train_rows,Validation_rows
0,1,2020-02-08 00:00:00-03:00,2022-02-06 00:00:00-03:00,2022-02-07 00:00:00-03:00,2023-02-06 00:00:00-03:00,585,250
1,2,2020-02-08 00:00:00-03:00,2023-02-06 00:00:00-03:00,2023-02-07 00:00:00-03:00,2024-02-06 00:00:00-03:00,835,214
2,3,2020-02-08 00:00:00-03:00,2024-02-06 00:00:00-03:00,2024-02-07 00:00:00-03:00,2025-02-05 00:00:00-03:00,1049,355
3,4,2020-02-08 00:00:00-03:00,2025-02-05 00:00:00-03:00,2025-02-06 00:00:00-03:00,2025-05-16 00:00:00-03:00,1404,95


## 5. Save the Feature Datasets

In [18]:
features_simple.to_csv(SIMPLE_FEATURES_OUTPUT, index=False)
features_final.to_csv(FINAL_FEATURES_OUTPUT, index=False)

print(
    f'Saved {len(features_simple):,} rows x {features_simple.shape[1]} columns '
    f'to {SIMPLE_FEATURES_OUTPUT}'
)
print(
    f'Saved {len(features_final):,} rows x {features_final.shape[1]} columns '
    f'to {FINAL_FEATURES_OUTPUT}'
)

Saved 1,499 rows x 19 columns to src/feat_eng/features_simple.csv
Saved 1,499 rows x 39 columns to src/feat_eng/features_final.csv


## Outputs

- `src/feat_eng/features_simple.csv`: calendar, causal gap indicators, and lagged total alarm volumes.
- `src/feat_eng/features_final.csv`: simple features plus conservatively available SELIC/IPCA values and lagged TR10/TR17/TR19 volumes.

In both datasets, `Date` identifies the target day, `recurrences_lt_30m` is the non-negative count target, and `target_available` records the quality rule used when preparing the model rows.